# Tuning: FD002 / FD004, XGBoost and LSTM

Exploration only, nothing here is load-bearing on its own (see CLAUDE.md) -- the winning hyperparameters found here get copied into `src/models.py`'s `XGB_HYPERPARAMS` / `LSTM_HYPERPARAMS` dicts, which *are* load-bearing.

**Why FD002/FD004 only:** both are above their published benchmark RMSE range for both model families (measured in `docs/Log.md`, 2026-09-10 session), while FD001/FD003 already land inside range with zero tuning. FD002/FD004 also have a genuinely harder learning problem -- 6 mixed operating regimes and (for FD004) 2 fault modes, vs. FD001/FD003's single condition -- so each gets tuned independently rather than searching for one hyperparameter setting shared across all four (decision discussed and confirmed: each dataset already trains a separate model instance, tuning just extends that to hyperparameters too).

**Selection metric:** GroupKFold CV RMSE for XGBoost (matches how every other XGBoost number in this project is validated, and `N_SPLITS=7` was just confirmed trustworthy in `eda_cv_folds.ipynb`), and validation-split RMSE for the LSTM (matches how the LSTM is already validated in `models.py` -- a single unit-based holdout, not k-fold, per the standing neural-net validation decision). Test-set RMSE at the final cycle is only computed *after* a winner is picked, never used to pick it -- picking hyperparameters by looking at the test set would be leakage of a different kind than the unit-split kind, but leakage all the same.

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

from src.features import add_rolling_features, add_rul_targets, add_savgol_features, find_constant_sensors
from src.load import SENSOR_COLS, load_train
from src.models import build_feature_cols, fit_lstm_scaler, make_xgb_model, train_lstm
from src.regimes import fit_regimes
from src.scoring import N_SPLITS, group_kfold_cv, rmse

SEED = 42
TUNE_DATASETS = ["FD002", "FD004"]


def build_train_features(dataset: str):
    train = load_train(dataset)
    train_norm, _ = fit_regimes(train)
    constant_sensors = find_constant_sensors(train_norm)
    varying_sensors = [s for s in SENSOR_COLS if s not in constant_sensors]
    train_feat = add_rolling_features(train_norm, varying_sensors)
    train_feat = add_savgol_features(train_feat, varying_sensors)
    train_feat = add_rul_targets(train_feat)
    feature_cols = build_feature_cols(varying_sensors)
    return train_feat, feature_cols


train_features = {ds: build_train_features(ds) for ds in TUNE_DATASETS}

## XGBoost: RandomizedSearchCV over the standard tree hyperparameters

**Question:** does searching over tree depth / learning rate / tree count / subsampling find something meaningfully better than XGBoost's plain defaults (what FD002/FD004 currently use)?

Grid ranges are the standard tuning ranges for gradient-boosted trees, not derived from EDA on this data specifically:
- `n_estimators`: [100, 200, 300, 500] -- more trees, more capacity, more compute
- `max_depth`: [3, 4, 5, 6, 8] -- default is 6; both shallower (less overfitting) and deeper (more capacity, since FD002/FD004 have more rows and more structure to fit) are worth trying
- `learning_rate`: [0.01, 0.03, 0.05, 0.1, 0.2] -- default is 0.3; smaller steps with more trees is the standard pairing
- `subsample` / `colsample_bytree`: [0.6, 0.8, 1.0] each -- row/column subsampling, a standard regularizer against overfitting
- `min_child_weight`: [1, 3, 5, 7] -- default is 1; higher values require more evidence before a tree splits further

`RandomizedSearchCV` samples 40 combinations from this space (seed=42, so the sample itself is reproducible) rather than the ~2,800 a full grid would need -- 40 is enough to cover the space reasonably given XGBoost trains fast, without an exhaustive search that isn't needed here.

In [2]:
XGB_PARAM_GRID = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 7],
}

xgb_search_results = {}
xgb_rows = []
for dataset in TUNE_DATASETS:
    train_feat, feature_cols = train_features[dataset]

    default_cv_rmses = group_kfold_cv(make_xgb_model, train_feat, feature_cols, "rul_capped", n_splits=N_SPLITS)
    default_mean = float(np.mean(default_cv_rmses))

    search = RandomizedSearchCV(
        estimator=make_xgb_model(),
        param_distributions=XGB_PARAM_GRID,
        n_iter=40,
        cv=GroupKFold(n_splits=N_SPLITS),
        scoring="neg_root_mean_squared_error",
        random_state=SEED,
        n_jobs=-1,
    )
    search.fit(train_feat[feature_cols], train_feat["rul_capped"], groups=train_feat["unit"])

    xgb_search_results[dataset] = search
    xgb_rows.append({
        "dataset": dataset,
        "default_cv_rmse": default_mean,
        "best_cv_rmse": -search.best_score_,
        "improvement": default_mean - (-search.best_score_),
        "best_params": search.best_params_,
    })

xgb_tuning_results = pd.DataFrame(xgb_rows)
xgb_tuning_results

,dataset,default_cv_rmse,best_cv_rmse,improvement,best_params
0,FD002,15.834899,14.894821,0.940078,"{'subsample': 0.6, 'n_estimators': 500, 'min_c..."
1,FD004,15.430032,14.652904,0.777128,"{'subsample': 0.8, 'n_estimators': 300, 'min_c..."


**XGBoost finding:** placeholder -- filled in after running the search above.

## LSTM: 4 reasoned variants vs. the current baseline

**Question:** does any of a small number of specific, explainable architecture changes beat the current LSTM (64 units, 1 layer, dropout 0.2, 30-cycle window) on validation RMSE?

Each variant is a genuine hypothesis about *why* FD002/FD004 specifically underperform, not a blind grid:
- **units=128**: 6 mixed operating regimes is a harder pattern than FD001/FD003's single condition -- maybe the model needs more capacity to represent it.
- **2 stacked layers (64 units each)**: lets the network build a regime-level representation in the first layer before a degradation-level one in the second, instead of learning both at once.
- **window=20**: FD002 test data has 6 units shorter than the current 30-cycle training window (minimum 21 cycles), and FD004 has 11 (minimum 19) -- see `docs/dataset-reference.md`. Training exclusively on 30-cycle windows means the model has never seen anything resembling what it faces on those short units at test time. A 20-cycle window closes most of that gap (fully for FD002, partially for FD004).
- **bidirectional**: added after a literature check specifically for this dataset (see `docs/Log.md`) turned up a published bidirectional-LSTM result beating plain and CNN-LSTM architectures on FD002. Reads each window back-to-front as well as front-to-back -- not future-peeking, since by the time the model sees a window it's already a fixed, fully-known slice of the past. Mechanically different from the other three (which all just add more of the same one-directional capacity), so it isn't a repeat of an already-disproven idea.

Selection metric is validation RMSE (the best epoch's `val_rmse`, i.e. what early stopping already tracks) -- not the test set, per the note above.

In [3]:
LSTM_VARIANTS = {
    "baseline (units=64, 1 layer, window=30)": {},
    "units=128": {"units": 128},
    "2 stacked layers (units=64 each)": {"num_layers": 2},
    "window=20": {"window": 20},
    "bidirectional": {"bidirectional": True},
}

lstm_models = {}
lstm_rows = []
for dataset in TUNE_DATASETS:
    train_feat, feature_cols = train_features[dataset]
    lstm_scaler = fit_lstm_scaler(train_feat, feature_cols)
    for variant_name, overrides in LSTM_VARIANTS.items():
        model = train_lstm(train_feat, feature_cols, lstm_scaler, **overrides)
        best_val_rmse = float(min(model.history.history["val_rmse"]))
        lstm_models[(dataset, variant_name)] = model
        lstm_rows.append({"dataset": dataset, "variant": variant_name, "val_rmse": best_val_rmse})

lstm_tuning_results = pd.DataFrame(lstm_rows)
lstm_tuning_results

Epoch 1/50


585/585 - 7s - 11ms/step - loss: 4597.5488 - rmse: 67.8052 - val_loss: 2565.6333 - val_rmse: 50.6521


Epoch 2/50


585/585 - 5s - 8ms/step - loss: 1858.2715 - rmse: 43.1077 - val_loss: 1063.9550 - val_rmse: 32.6183


Epoch 3/50


...

Epoch 9/50


681/681 - 7s - 10ms/step - loss: 43.6788 - rmse: 6.6090 - val_loss: 305.4193 - val_rmse: 17.4762


Epoch 10/50


681/681 - 10s - 15ms/step - loss: 39.1896 - rmse: 6.2602 - val_loss: 318.3306 - val_rmse: 17.8418


,dataset,variant,val_rmse
0,FD002,"baseline (units=64, 1 layer, window=30)",14.661711
1,FD002,units=128,13.299772
2,FD002,2 stacked layers (units=64 each),14.633920
3,FD002,window=20,15.439909
4,FD002,bidirectional,15.234427
5,FD004,"baseline (units=64, 1 layer, window=30)",15.982450
6,FD004,units=128,16.468618
7,FD004,2 stacked layers (units=64 each),17.579714
8,FD004,window=20,16.673950
9,FD004,bidirectional,16.765636


In [4]:
print("val_rmse by dataset x variant:")
print(lstm_tuning_results.pivot(index="variant", columns="dataset", values="val_rmse").round(2))
print()

for dataset in TUNE_DATASETS:
    subset = lstm_tuning_results[lstm_tuning_results["dataset"] == dataset]
    winner = subset.loc[subset["val_rmse"].idxmin(), "variant"]
    print(f"{dataset}: best variant = {winner} (val_rmse={subset['val_rmse'].min():.2f})")

val_rmse by dataset x variant:
dataset                                  FD002  FD004
variant                                              
2 stacked layers (units=64 each)         14.63  17.58
baseline (units=64, 1 layer, window=30)  14.66  15.98
bidirectional                            15.23  16.77
units=128                                13.30  16.47
window=20                                15.44  16.67

FD002: best variant = units=128 (val_rmse=13.30)
FD004: best variant = baseline (units=64, 1 layer, window=30) (val_rmse=15.98)


**LSTM finding:** placeholder -- filled in after running the variants above.

## Test-set check (informational only -- not used to pick anything above)

Evaluates the winning XGBoost and LSTM configuration per dataset at the final test cycle, to see the actual required-output-shaped number and compare against the current (untuned) test RMSEs on record: FD002 xgboost=28.73/lstm=27.86, FD004 xgboost=28.82/lstm=26.53 (`docs/Log.md`, 2026-09-12).

In [5]:
from src.load import load_rul, load_test
from src.regimes import apply_regimes
from src.scoring import evaluate_at_final_cycle, evaluate_sequence_model_at_final_cycle

for dataset in TUNE_DATASETS:
    train_feat, feature_cols = train_features[dataset]
    test = load_test(dataset)
    rul_true = load_rul(dataset)

    _, fitted = fit_regimes(load_train(dataset))
    test_norm = apply_regimes(test, fitted)
    constant_sensors = find_constant_sensors(fit_regimes(load_train(dataset))[0])
    varying_sensors = [s for s in SENSOR_COLS if s not in constant_sensors]
    test_feat = add_rolling_features(test_norm, varying_sensors)
    test_feat = add_savgol_features(test_feat, varying_sensors)

    xgb_best_model = xgb_search_results[dataset].best_estimator_
    xgb_test = evaluate_at_final_cycle(xgb_best_model, test_feat, feature_cols, rul_true)
    print(f"{dataset} xgboost tuned: test rmse={xgb_test['rmse']:.2f} (was {'28.73' if dataset=='FD002' else '28.82'})")

    subset = lstm_tuning_results[lstm_tuning_results["dataset"] == dataset]
    winner = subset.loc[subset["val_rmse"].idxmin(), "variant"]
    lstm_best_model = lstm_models[(dataset, winner)]
    lstm_scaler = fit_lstm_scaler(train_feat, feature_cols)
    test_feat_scaled = test_feat.copy()
    test_feat_scaled[feature_cols] = lstm_scaler.transform(test_feat[feature_cols])
    window_used = LSTM_VARIANTS[winner].get("window", 30)
    lstm_test = evaluate_sequence_model_at_final_cycle(lstm_best_model, test_feat_scaled, feature_cols, rul_true, window_used)
    print(f"{dataset} lstm tuned ({winner}): test rmse={lstm_test['rmse']:.2f} (was {'27.86' if dataset=='FD002' else '26.53'})")

FD002 xgboost tuned: test rmse=27.71 (was 28.73)


FD002 lstm tuned (units=128): test rmse=26.10 (was 27.86)


FD004 xgboost tuned: test rmse=28.24 (was 28.82)


FD004 lstm tuned (baseline (units=64, 1 layer, window=30)): test rmse=27.72 (was 26.53)


**Overall finding:** placeholder -- filled in after running everything above. Whatever wins gets copied into `src/models.py`'s `XGB_HYPERPARAMS` / `LSTM_HYPERPARAMS` dicts for FD002/FD004, and `src/pipeline.py` gets updated to use them.